In [24]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 32
max_iters = 10000
# eval_interval = 2500
learning_rate = 1e-3
eval_iters = 250

cuda


In [25]:
with open('wizard_of_oz.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()

In [26]:
print("length of dataset in characters: ", len(text))
print(text[:200])

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)
print(vocab_size)

length of dataset in characters:  232309
﻿  DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NEW
['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']
81


In [27]:
str_to_int = {char : idx for idx, char in enumerate(chars)} # creatd this dict to map chars to ints
int_to_str = {idx : char for idx, char in enumerate(chars)} # created this dict to map ints to chars

encode = lambda string: [str_to_int[c] for c in string] # its iterating char from the string and converting the char to an int based on the str to int mapping
decode = lambda lst: ''.join([int_to_str[i] for i in lst]) # its taking list of ints and returning strings

print(f"str to int: {str_to_int}")
print(f"int to str: {int_to_str}")
print("")
print(encode("Hello"))
print(decode(encode("Hello")))


str to int: {'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, ';': 23, '?': 24, 'A': 25, 'B': 26, 'C': 27, 'D': 28, 'E': 29, 'F': 30, 'G': 31, 'H': 32, 'I': 33, 'J': 34, 'K': 35, 'L': 36, 'M': 37, 'N': 38, 'O': 39, 'P': 40, 'Q': 41, 'R': 42, 'S': 43, 'T': 44, 'U': 45, 'V': 46, 'W': 47, 'X': 48, 'Y': 49, 'Z': 50, '[': 51, ']': 52, '_': 53, 'a': 54, 'b': 55, 'c': 56, 'd': 57, 'e': 58, 'f': 59, 'g': 60, 'h': 61, 'i': 62, 'j': 63, 'k': 64, 'l': 65, 'm': 66, 'n': 67, 'o': 68, 'p': 69, 'q': 70, 'r': 71, 's': 72, 't': 73, 'u': 74, 'v': 75, 'w': 76, 'x': 77, 'y': 78, 'z': 79, '\ufeff': 80}
int to str: {0: '\n', 1: ' ', 2: '!', 3: '"', 4: '&', 5: "'", 6: '(', 7: ')', 8: '*', 9: ',', 10: '-', 11: '.', 12: '0', 13: '1', 14: '2', 15: '3', 16: '4', 17: '5', 18: '6', 19: '7', 20: '8', 21: '9', 22: ':', 23: ';', 24: '?', 25: 'A', 26: 'B', 27: 'C', 28: 'D

In [28]:
data = torch.tensor(encode(text), dtype= torch.long) #tokenized or encoded the entire .txt file and then fed to pytorch to return the entire dataset as tensor
print(data[:100])

n = int(0.8 * len(data))
train_data = data[:n]
validation_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else validation_data
    idx = torch.randint(len(data) - block_size, (batch_size,))

    #print(idx)

    x = torch.stack([data[i: i + block_size] for i in idx]).to(device)
    y = torch.stack([data[i+1: i + block_size + 1] for i in idx]).to(device)

    return x, y

x, y = get_batch('train')
print('inputs: ' )
print(x)
print('targets: ')
print(y)


tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])
inputs: 
tensor([[78, 11,  1,  3, 26, 74, 73,  1],
        [57, 71, 58, 76,  1, 74, 69,  1],
        [44, 61, 58,  1, 31, 54, 71, 60],
        [51, 33, 65, 65, 74, 72, 73, 71],
        [ 1, 67, 54, 73, 74, 71, 54, 65],
        [58, 54, 64,  0, 73, 61, 71, 68],
        [61, 62, 72,  1, 62, 71, 68, 67],
        [68, 65, 75, 58, 71,  1, 66, 54],
        [73,  1, 76, 54, 72,  1, 73, 61],
        [72,  1, 73, 68,  1, 60, 58, 73],
        [ 1, 27, 45, 44,  1, 44, 32, 29],
        [58, 57, 62, 54, 73, 58, 65, 78],
        [ 1, 54, 72, 64, 58, 57,  1, 3

In [29]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [30]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


Z:l 99[Y﻿8.gycT&zlAXc5-6zEDkY8D5t;YsKlFvyZ!m8&iGCnSpF]pbLdOC2cxk3,z8F1yDs-]wn 9p7-1P﻿﻿YN"Wx3pTm !7PRvbyb)xc0EaXV*FWx7.dVqQMMH?4.h'OpBm:ogRF]MabE4bQnqRx
Ac;h
m3T;m4U5YZ3i(oGv?um?E5 !VHQo)9KD apXqaIK6.C:!J.vAwNwwdMSZGD_Rpv qTK
bq6!
b2hbXtu"WP5p2kd-mai&0SZ1]tEW7MRe''wH,uQ2IXqjU5"[zNH]qAdM,R1Ua-gRV_i115rR1:r[wUJ)VLLld5YU7P2xKIoi37-
bz
!4!
,n!ES6Yd
SwUhO7Bmy0-mB2ves3i(E?
h  R(xYJwQ)Jxx;q6O9x_"[:1 coJUonqw*k,oH,Ogy0CSyS   :q8&Zo"cjtrDE g0ET3Z&
bE9Q*yxiU0x7
4ooQQs﻿2Mh(nt .0AlT*1lTUl
nBm)DDN"vo08PQGX8pu


In [31]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.791, val loss: 4.799
step: 250, train loss: 4.491, val loss: 4.511
step: 500, train loss: 4.221, val loss: 4.237
step: 750, train loss: 3.973, val loss: 3.992
step: 1000, train loss: 3.752, val loss: 3.776
step: 1250, train loss: 3.576, val loss: 3.595
step: 1500, train loss: 3.415, val loss: 3.429
step: 1750, train loss: 3.269, val loss: 3.291
step: 2000, train loss: 3.157, val loss: 3.171
step: 2250, train loss: 3.046, val loss: 3.071
step: 2500, train loss: 2.959, val loss: 2.984
step: 2750, train loss: 2.878, val loss: 2.915
step: 3000, train loss: 2.824, val loss: 2.851
step: 3250, train loss: 2.766, val loss: 2.796
step: 3500, train loss: 2.715, val loss: 2.758
step: 3750, train loss: 2.679, val loss: 2.710
step: 4000, train loss: 2.651, val loss: 2.681
step: 4250, train loss: 2.613, val loss: 2.652
step: 4500, train loss: 2.581, val loss: 2.615
step: 4750, train loss: 2.558, val loss: 2.610
step: 5000, train loss: 2.550, val loss: 2.593
step: 5250, train l

In [32]:
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


The itatr woruived bithed Fode pithed bo fod f m'toom!"Hiftousiz5Yo ggngnd.


f Zedito abutingouthe e. je toould ablet aigam
Ar


"
athinsteds akan hat!"Wxcemee granternge wste, aintaunshare wo sowhethi; w pl

Mgatio foulocat; Zere cen.

bomf f hid "I
BLOf eim tin ie, adine hesm clarewan FLard aty  lld te t parinved
CIsir tshed wan wewey upped  vere be chey te m g ck!"
" pe*
aney  drorinaI Silise
"Soor thasthin t bo, ansoo, wsosehas ig quneres on t aner bedund crkld atot DEBut bshfNoferledm thes
